# Notebook 1: collect, clean and prepare the data

**Workshop: from plant data to a trained model.** Repository: `water_resource`.

We take the official influent data of the BSM1 wastewater plant (what flows into the plant, every 15 minutes, for 14 days) and turn it into a dataset a model can learn from:

| Part | What we do | Where the repository does it |
|---|---|---|
| **A. Collect** | download, open, name the columns, first look | `src/wwtp/official/plant.py` (`_official_table`) |
| **B. Clean** | check, find and fix problems, physics checks, one time grid | `plant.py` (`load_influent`, `official_series`), `docs/DATASET.md` |
| **C. Prepare** | choose inputs and target, split by time, transform, scale, cut windows, save | `src/wwtp/forecast/dataset.py`, `src/wwtp/warning/dataset.py` |

**The task we prepare for:** forecast the inflow for the next 2 hours from the last 24 hours, as the repository's influent forecaster does (`forecast/dataset.py`: `LOOKBACK = 96`, `HORIZON = 8`).

Run the cells from top to bottom with **Shift + Enter**. Cells marked ✏️ are for you to change and re-run.

In [ ]:
from pathlib import Path
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("../data")          # workshop/data/ (not committed to git)
DATA.mkdir(exist_ok=True)
plt.rcParams.update({"figure.figsize": (11, 3.2), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 140)

---
## A. Collect

### A1. Download, with the source pinned

The three IWA influent files (dry, rain and storm weather) are public spreadsheets on GitHub. We download them **from a fixed commit**: the URL then always returns the same bytes, and anyone can rebuild our dataset. This is exactly the URL in `src/wwtp/official/plant.py`.

The files are GPL-licensed, so the repository does not copy them: it downloads them on first use. We do the same and keep them in `workshop/data/`.

In [ ]:
REV = "5dadb209a5a34ecb0e2f83a2e79c060bf3470582"      # pinned commit (same as plant.py)
URL = "https://raw.githubusercontent.com/davidfernand/BSM1_in_Python/{rev}/code/Influent%20files/{name}"
FILES = {"dry": "Inf_dry_2006.xlsx", "rain": "Inf_rain_2006.xlsx", "storm": "Inf_strm_2006.xlsx"}


def download(weather: str) -> Path:
    """Download one file the first time; afterwards use the local copy."""
    path = DATA / FILES[weather]
    if not path.exists():
        print("downloading", FILES[weather])
        urllib.request.urlretrieve(URL.format(rev=REV, name=FILES[weather]), path)
    return path


paths = {w: download(w) for w in FILES}
for w, p in paths.items():
    print(f"{w:6s} {p}  {p.stat().st_size / 1024:.0f} kB")

### A2. Open the raw file and **look** at it before doing anything

In [ ]:
raw = pd.read_excel(paths["dry"], header=None)
print("shape:", raw.shape)
raw.head(4)

In [ ]:
raw.dtypes.value_counts()

**What we see:**
- Row 0 is a header with names, not data. Because of it, pandas made every column `object` (text) instead of numbers.
- 1346 rows = 1 header + 1345 samples: 14 days × 96 samples per day + 1.

The repository solves this by keeping only rows whose first cell is a number (`plant.py:44`). In pandas we can simply say "row 0 is the header". We also give the columns clear names and **write the units down**: the file does not contain them.

In [ ]:
COLS = ["t", "S_I", "S_S", "X_I", "X_S", "X_BH", "X_BA", "X_P", "S_O", "S_NO", "S_NH", "S_ND", "X_ND", "S_ALK", "Q"]
UNITS = {"t": "d", "Q": "m3/d", "S_ALK": "mol/m3", **{c: "g/m3" for c in COLS[1:13]}}
MEANING = {"t": "time", "S_I": "inert soluble COD", "S_S": "readily biodegradable substrate (COD)",
           "X_I": "inert particulate COD", "X_S": "slowly biodegradable substrate", "X_BH": "heterotrophic biomass",
           "X_BA": "autotrophic (nitrifying) biomass", "X_P": "particulate products of decay", "S_O": "oxygen",
           "S_NO": "nitrate", "S_NH": "ammonium", "S_ND": "soluble organic nitrogen",
           "X_ND": "particulate organic nitrogen", "S_ALK": "alkalinity", "Q": "flow rate"}


def load(weather: str) -> pd.DataFrame:
    d = pd.read_excel(paths[weather], header=0)
    d.columns = COLS
    return d.astype(float)


df = {w: load(w) for w in FILES}
pd.DataFrame({"meaning": MEANING, "unit": UNITS})

In [ ]:
df["dry"].describe().T.round(2)

### A3. First checks: size, time step, missing values, duplicates

Always check these, on every dataset.

In [ ]:
for w, d in df.items():
    step_min = np.round(np.diff(d.t) * 1440, 3)
    print(f"{w:6s} rows {len(d)}  days {d.t.min():.0f}-{d.t.max():.0f}  "
          f"step(s) in minutes {np.unique(step_min)}  NaN {d.isna().sum().sum()}  "
          f"duplicate times {d.t.duplicated().sum()}  negative values {(d.drop(columns='t') < 0).sum().sum()}")

Clean so far: a constant 15-minute step and no gaps. One detail: the last row (t = 14) repeats the first one (t = 0), because the file is periodic. When we cut windows later we use t < 14 only.

In [ ]:
d = df["dry"]
print("first row == last row (apart from time)?", np.allclose(d.iloc[0, 1:], d.iloc[-1, 1:], atol=1e-3))

### A4. Plot the data

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
for w, c in zip(FILES, ["tab:blue", "tab:green", "tab:purple"]):
    axes[0].plot(df[w].t, df[w].Q, lw=1, color=c, label=w)
    axes[1].plot(df[w].t, df[w].S_NH, lw=1, color=c, label=w)
axes[0].set_ylabel("inflow Q [m³/d]")
axes[1].set_ylabel("ammonium S_NH [g/m³]")
axes[1].set_xlabel("day")
axes[0].legend(ncol=3)
plt.tight_layout()

**Questions:** Where is the daily cycle? What do weekends look like (days 5–6 and 12–13)? What does rain do to the ammonium *concentration*, and why?

The three curves look identical in the first week. Let's check this, not guess it:

In [ ]:
week1 = df["dry"].t < 7
for w in ("rain", "storm"):
    diff = (df[w][week1] - df["dry"][week1]).abs().drop(columns=["t", "X_ND"]).max().max()
    print(f"largest difference dry vs {w} in week 1: {diff:.4f}")

**Week 1 is the same dry week in all three files.** If we trained on rain week 1 and tested on dry week 1, we would test on training data. This is why paper 2 learns on week 1 and tests on week 2, and why we split by time in part C.

---
## B. Clean

The official files are tidy. Real plant data never is. To practise, we **damage a copy** the way real sensors do, then detect and repair it, and compare with the truth (which we have here, and normally don't).

In [ ]:
def damage(d: pd.DataFrame, seed: int = 0) -> pd.DataFrame:
    """Add typical sensor problems to a copy of the data."""
    rng = np.random.default_rng(seed)
    x = d.copy()
    x.loc[rng.choice(len(x), 6, replace=False), "S_NH"] = np.nan             # single missing samples
    x.loc[400:405, ["S_NH", "Q"]] = np.nan                                   # a 1.5-hour gap
    x.loc[[150, 700, 1100], "S_NH"] *= 10                                    # spikes (unit / typing errors)
    x.loc[900:915, "S_NH"] = x.loc[900, "S_NH"]                              # sensor stuck for 4 h
    x.loc[1200, "Q"] = -x.loc[1200, "Q"]                                     # impossible negative flow
    x = pd.concat([x, x.iloc[[300, 301]]])                                   # duplicated rows
    return x.sample(frac=1, random_state=seed).reset_index(drop=True)        # rows out of order


dirty = damage(df["dry"])
truth = df["dry"]
print(len(dirty), "rows")
dirty.head()

### B1. Detect

In [ ]:
print("time sorted?           ", dirty.t.is_monotonic_increasing)
print("duplicate times        ", dirty.t.duplicated().sum())
print("missing values per col \n", dirty.isna().sum()[lambda s: s > 0])
print("negative values        ", (dirty[["Q", "S_NH"]] < 0).sum().to_dict())

### B2. Repair, step by step

1. sort by time and drop duplicate timestamps;
2. put the data on the full 15-minute grid, so that missing *rows* become visible NaN;
3. physically impossible values (negative flow) → NaN;
4. spikes: compare each value with a rolling median; far away (in units of the median absolute deviation, MAD) → NaN;
5. stuck sensor: zero variation for 2 hours or more → NaN;
6. fill **short** gaps (≤ 1 h) by interpolation in time; leave longer gaps as NaN and drop them later. (Pitfall: pandas' `interpolate(limit=4)` fills the first 4 samples of *every* gap, long ones too.)

⚠️ The rolling median below is **centred**: it uses samples after the one it checks. That is fine for cleaning a stored dataset, but a controller running live has no future samples: there, use a trailing window.

In [ ]:
def clean(x: pd.DataFrame, cols=("Q", "S_NH"), max_gap=4) -> tuple[pd.DataFrame, pd.DataFrame]:
    flags = {}
    x = x.sort_values("t").drop_duplicates("t").reset_index(drop=True)              # 1
    grid = np.round(np.arange(0, 14 + 1e-9, 15 / 1440), 6)                          # 2
    x = x.set_index(np.round(x.t, 6)).reindex(grid).rename_axis("t_grid").reset_index()
    x["t"] = x["t_grid"]
    x = x.drop(columns="t_grid")
    for c in cols:
        bad_neg = x[c] < 0                                                          # 3
        med = x[c].rolling(9, center=True, min_periods=3).median()                 # 4
        mad = (x[c] - med).abs().rolling(9, center=True, min_periods=3).median()
        bad_spike = (x[c] - med).abs() > 6 * (mad + 1e-6) + 0.5 * med.abs()
        stuck = x[c].diff().abs().rolling(8).max() == 0                             # 5
        bad_stuck = stuck.copy()
        for k in range(1, 8):
            bad_stuck |= stuck.shift(-k, fill_value=False)
        flags[c] = pd.DataFrame({"missing": x[c].isna(), "negative": bad_neg, "spike": bad_spike, "stuck": bad_stuck})
        x.loc[bad_neg | bad_spike | bad_stuck, c] = np.nan
        x[c] = fill_short_gaps(x[c], max_gap)                                       # 6
    return x, pd.concat(flags, axis=1)


def fill_short_gaps(s: pd.Series, max_gap: int) -> pd.Series:
    """Interpolate gaps of at most ``max_gap`` samples; leave longer gaps empty.

    Careful: ``s.interpolate(limit=4)`` would also fill the first 4 samples of a
    long gap, inventing data. Here each gap's full length is measured first."""
    isna = s.isna()
    gap_len = isna.groupby((~isna).cumsum()).transform("sum")
    return s.where(~isna | (gap_len > max_gap), s.interpolate(limit_area="inside"))


fixed, flags = clean(dirty)
print(flags.sum().unstack())
print("\nNaN left after repair:", fixed[["Q", "S_NH"]].isna().sum().to_dict())

How good was the repair? We know the truth here, so we can measure it:

In [ ]:
err = (fixed[["Q", "S_NH"]] - truth[["Q", "S_NH"]]).abs()
print("largest error after repair (where a value exists):")
print(err.max().round(3))

fig, ax = plt.subplots()
sel = slice(120, 1150)
ax.plot(dirty.sort_values("t").t, dirty.sort_values("t").S_NH, color="tab:red", lw=1, label="damaged")
ax.plot(truth.t, truth.S_NH, color="k", lw=2, alpha=.35, label="truth")
ax.plot(fixed.t, fixed.S_NH, color="tab:blue", lw=1, label="repaired")
ax.set_ylim(0, 60); ax.set_ylabel("S_NH [g/m³]"); ax.set_xlabel("day"); ax.legend()

✏️ **Try:** change the spike threshold (`6 *`) to `2 *`. How many good values are now thrown away? Cleaning rules have false alarms too.

### B3. Physics checks and corrections

Statistics can't find this one; physics can. In the spreadsheets the inert COD `S_I` is 30 g/m³ in **every** row, also when rain doubles the flow. But rain water contains no COD, so it must **dilute** it. The official rain file shipped with `bsm2-python` follows `S_I = 30 · Q_dry / Q`. The repository applies this rule to rain and storm (`plant.py:89`) and documents it (`docs/DATASET.md` §3.3).

The model also needs a column the file does not have: total suspended solids, `TSS = 0.75 (X_I + X_S + X_BH + X_BA + X_P)` (`plant.py:90`).

In [ ]:
for w in FILES:
    print(f"{w:6s} S_I unique values: {df[w].S_I.unique()[:5]}")

for w in FILES:
    df[w]["S_I"] = 30.0 * df["dry"].Q.values / df[w].Q.values
    df[w]["TSS"] = 0.75 * df[w][["X_I", "X_S", "X_BH", "X_BA", "X_P"]].sum(axis=1)

ax = df["rain"].plot(x="t", y="S_I", legend=False, lw=1, title="S_I in the rain file after the dilution fix")
ax.set_ylabel("g/m³")

### B4. One time grid

The plant model runs every 45 s; the influent is sampled every 15 min. Between samples the value is held (zero-order hold): each sample is used for 20 control steps. This is what `official_series` does in `plant.py:144`.

In [ ]:
q = df["storm"].set_index(pd.to_timedelta(df["storm"].t, unit="D"))["Q"]
q45 = q.resample("45s").ffill()
print("15-min samples:", len(q), "  45-s steps:", len(q45), " (14 days × 1920 steps per day + 1)")
q45.iloc[:45].plot(drawstyle="steps-post", marker=".", title="first 30 minutes: held constant between samples")

---
## C. Prepare

### C1. Inputs and target

**Inputs:** only what a real plant can measure at the inlet: flow `Q`, ammonium `S_NH` and readily biodegradable COD `S_S` (the same three channels as `forecast/dataset.py`: `CHANNELS = ("Q_in", "S_NH_in", "S_S_in")`).

**Target:** the flow over the next 2 hours (8 samples).

**Shapes:** one example is `x` = (96 time steps, 3 channels) → `y` = (8,).

In [ ]:
CHANNELS = ["Q", "S_NH", "S_S"]
LOOKBACK, HORIZON = 96, 8           # 24 h in, 2 h out, at 15 min

### C2. Split by time

Neighbouring windows share most of their samples, so a random split would put near-copies of training examples into the test set. We split by **when the target happens**:

| split | data | contains |
|---|---|---|
| train | dry, all 14 days; rain, targets in days 7–14 | daily cycles, weekends, one long rain event |
| validation | storm, targets in days 7–10.25 | storm 1 (day 8.8) |
| test | storm, targets in days 10.25–14 | storm 2 (day 11) |

A window's 24-hour *input* may start before its split (inputs are the past and always available); its *target* must lie fully inside the split. The model never sees a storm in training: the test asks whether it generalises.

In [ ]:
SPLITS = {"train": [("dry", 1.0, 14.0), ("rain", 7.0, 14.0)],
          "val": [("storm", 7.0, 10.25)],
          "test": [("storm", 10.25, 14.0)]}

### C3. Transform and scale, **with training statistics only**

1. `log`: flows and loads change by factors; a storm then doesn't dominate the scale.
2. standardise: `z = (x − mean) / std` per channel, so all inputs sit around 0 with spread 1.

The mean and std are computed on the training data **only** and then used everywhere (validation, test, deployment), exactly like `Normaliser.fit` in `warning/dataset.py:239`.

In [ ]:
def rows_of(weather, t_lo, t_hi):
    """Raw samples whose time lies in [t_lo, t_hi) (the training rows used for the statistics)."""
    d = df[weather]
    return d[(d.t >= t_lo) & (d.t < t_hi)][CHANNELS].values

# every sample the training windows touch: their targets and their 24-h history
train_rows = np.concatenate([rows_of(w, lo - 1.0, hi) for w, lo, hi in SPLITS["train"]])
log_train = np.log(train_rows)
MEAN, STD = log_train.mean(0), log_train.std(0)
print("training rows:", len(train_rows))
print("mean of log:", MEAN.round(3), "  std of log:", STD.round(3))

# what if we (wrongly) used all the data, test included?
all_rows = np.concatenate([df[w][CHANNELS].values for w in FILES])
print("with the test data included the mean would be", np.log(all_rows).mean(0).round(3), " ← information from the test set")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 2.6))
axes[0].hist(train_rows[:, 0], 40); axes[0].set_title("Q raw [m³/d]")
axes[1].hist(np.log(train_rows[:, 0]), 40, color="tab:green"); axes[1].set_title("log Q")
axes[2].hist((np.log(train_rows[:, 0]) - MEAN[0]) / STD[0], 40, color="tab:orange"); axes[2].set_title("standardised: mean 0, std 1")
plt.tight_layout()

### C4. Cut windows

The same idea as `_windows` in `forecast/dataset.py:67`: input = the 96 samples up to "now", target = the next 8 flow values. We assert that the target starts strictly after the input ends.

In [ ]:
def scale(x):
    return (np.log(x) - MEAN) / STD


def windows(weather, t_lo, t_hi, stride=1):
    d = df[weather][df[weather].t < 14.0 - 1e-9]       # drop the repeated last row
    z = scale(d[CHANNELS].values).astype(np.float32)
    t = d.t.values
    X, Y, T = [], [], []
    for s in range(0, len(d) - LOOKBACK - HORIZON + 1, stride):
        first, last = s + LOOKBACK, s + LOOKBACK + HORIZON - 1      # target rows
        if t[first] < t_lo - 1e-9 or t[last] >= t_hi - 1e-9:
            continue
        X.append(z[s:s + LOOKBACK])                    # past
        Y.append(z[first:last + 1, 0])                 # future flow (channel 0)
        T.append(t[first])
        assert s + LOOKBACK - 1 < first                # causal: input ends before the target starts
    return np.array(X), np.array(Y), np.array(T)


data = {}
for split, parts in SPLITS.items():
    xs, ys, ts = zip(*[windows(*p) for p in parts])
    data[split] = (np.concatenate(xs), np.concatenate(ys), np.concatenate(ts))
    print(f"{split:5s}  X {data[split][0].shape}  y {data[split][1].shape}")

Read the shapes aloud: **(N examples, 96 time steps, 3 channels) → (N examples, 8 future values)**. This shape decides which models we can use (notebook 2).

One example, back in physical units:

In [ ]:
X, Y, T = data["val"]
i = int(np.argmin(np.abs(T - 8.7)))                    # just before storm 1
to_flow = lambda z: np.exp(z * STD[0] + MEAN[0])
fig, ax = plt.subplots()
ax.plot(np.arange(-LOOKBACK, 0) / 4, to_flow(X[i][:, 0]), label="input x: last 24 h")
ax.plot(np.arange(0, HORIZON) / 4, to_flow(Y[i]), "o-", color="tab:orange", label="target y: next 2 h")
ax.axvline(0, color="k", lw=1, ls="--")
ax.set_xlabel("hours from now"); ax.set_ylabel("Q [m³/d]"); ax.legend(); ax.set_title(f"one validation example, day {T[i]:.2f}")

### C5. Save, with everything needed to use the data again

The scaler (mean, std) travels with the data. A model trained on scaled data is useless without it.

In [ ]:
np.savez_compressed(
    DATA / "prepared.npz",
    **{f"X_{s}": data[s][0] for s in data}, **{f"y_{s}": data[s][1] for s in data},
    **{f"t_{s}": data[s][2] for s in data},
    mean=MEAN, std=STD, channels=np.array(CHANNELS), lookback=LOOKBACK, horizon=HORIZON, source_rev=REV)
print("saved", DATA / "prepared.npz")

### ✅ Checklist: what you just did

- [x] downloaded from a pinned source, cached locally
- [x] looked at the raw file before parsing it, named the columns, wrote the units down
- [x] checked time step, gaps, duplicates, impossible values
- [x] repaired a damaged copy and measured how good the repair was
- [x] a physics check found an error statistics could not (`S_I`)
- [x] split **by time**, scaled with **training statistics only**, cut causal windows
- [x] saved data + scaler together

**Next:** notebook 2, choose and train a model.